<a href="https://colab.research.google.com/github/hoki621/tbf21-gemma4-mhealth/blob/main/Gemma4_MHEALTH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemma 4 12Bを使ってセンサーデータから人間行動認識をしてみた

MHEALTHの2秒間の波形から12種類の活動コードを予測します。
100時点×15チャネルを10個のpatchに分け、線形投影でGemmaの入力embeddingへ挿入し、projectorとLoRAを学習します。

BF16対応のGPU runtimeで、データ準備、モデル読込、学習、評価の順に実行してください。
学習ループにはTransformers Trainerを使います。Gemma本体はBF16で凍結し、projectorはFP32で学習します。

In [1]:
%pip install -q transformers==5.20.2 peft==0.21.2 accelerate==1.15.0 safetensors==0.8.0 numpy==2.2.6 polars==1.44.2
# インストール後にランタイムを再起動し、このセルを飛ばして続けてください。


In [2]:
%pip uninstall -y torchao

モデルの利用条件を確認し、認証が必要な場合はHugging Faceのread権限でログインします。Colab Secretsの `HF_TOKEN` を利用し、tokenをノートブックへ直接記載しません。外部への学習ログ送信やモデル公開は行いません。


In [3]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

In [4]:
import hashlib, math, re, urllib.request, zipfile
from functools import cache
from pathlib import Path
import numpy as np
import polars as pl
import torch
from torch import nn
from torch.nn import functional as F
from transformers import Gemma4UnifiedForConditionalGeneration, Trainer, TrainingArguments, EarlyStoppingCallback, set_seed
from transformers import AutoTokenizer, GenerationConfig, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model
from sklearn.metrics import classification_report

MODEL_ID = "google/gemma-4-12B-it"
REVISION = "707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7"
OUTPUT = Path("gemma4_mhealth_run")
OUTPUT.mkdir(exist_ok=True)

設定するのはHugging Faceの認証と保存先 `OUTPUT` です。データzipは作業ディレクトリへ自動取得します。
以下の `archive` を既存zipのパスに変更しても構いません。出力はColabの一時ディスクなので、終了前に保存先フォルダをダウンロードしてください。

## 1. MHEALTHを準備する（CPU）

SensorLLMの列選択・連続ラベル区間・窓切りをMITライセンスで改変利用します。
原典との差は、**末尾窓を追加しない、被験者10をvalに留保、評価stride100**です。
trainは2/4/5/7/8/9、testは1/3/6。100時点×15ch、50Hzの2秒窓で、label0と混在窓を除きます。
正規化統計はtrainのraw行のみ（label0含む）、FP64の二段階計算後にFP32へ戻します。

In [5]:
archive = Path("mhealth+dataset.zip")
if not archive.exists():
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/319/mhealth%2Bdataset.zip", archive)
with archive.open("rb") as stream:
    if hashlib.file_digest(stream, "sha256").hexdigest() != "16ad0ce709f3f00df18f348610d15bce0884b79e2143f57f446493673f02b8e0":
        raise ValueError("MHEALTH zipのSHA256が異なります。")
columns = (0, 1, 2, 5, 6, 7, 8, 9, 10, 14, 15, 16, 17, 18, 19)
PATCH_SAMPLES = 10
PATCH_WIDTH = PATCH_SAMPLES * len(columns)
train_subjects = (2, 4, 5, 7, 8, 9)

@cache
def read_subject(subject: int, usecols: tuple) -> np.ndarray:
    with zipfile.ZipFile(archive) as source:
        return np.loadtxt(source.open(f"MHEALTHDATASET/mHealth_subject{subject}.log"), usecols=usecols, dtype=np.float32)

# SensorLLMのsplit_sequencesを改変: 連続区間ごとに100時点、末尾の追加窓は作らない。
def windows(subject: int, stride: int) -> list[tuple[int, int, int]]:
    labels = read_subject(subject, (23,))
    boundaries = np.r_[0, np.flatnonzero(labels[1:] != labels[:-1]) + 1, len(labels)]
    result = []
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        if labels[start] != 0:
            result.extend((subject, int(labels[start]), i) for i in range(int(start), int(end)-99, stride))
    return result

arrays = [read_subject(s, columns).astype(np.float64) for s in train_subjects]
raw_rows = sum(map(len, arrays))
mean = sum(a.sum(0) for a in arrays) / raw_rows
std = np.sqrt(sum(((a-mean)**2).sum(0) for a in arrays) / raw_rows).clip(1e-6)
mean, std = mean.astype(np.float32), std.astype(np.float32)
train_meta = [w for s in train_subjects for w in windows(s, 50)]
val_meta = windows(10, 100)
TRAIN_WEIGHTS = len(train_meta) / (12*np.bincount([w[1] for w in train_meta], minlength=13)[1:])
VAL_WEIGHTS = len(val_meta) / (12*np.bincount([w[1] for w in val_meta], minlength=13)[1:])

def preprocess_window(window: tuple[int, int, int]) -> dict:
    subject, label, start = window
    return {"waveforms": (read_subject(subject, columns)[start:start+100]-mean)/std, "labels": label}

np.savez(OUTPUT / "preprocessing.npz", mean=mean, std=std, train=train_meta, validation=val_meta)
print(f"train={len(train_meta)}, val={len(val_meta)}, 正規化raw行={raw_rows}")

train=4024, val=329, 正規化raw行=735745


期待値はtrain4,024窓、val329窓、正規化735,745行です。test波形はまだ読みません。
以下は採用済みのpromptです。markerを10個の埋め込みへ置き換えます。語彙のembedding行は増やしません。

In [6]:
CLASS_NAMES = (
    "Standing still", "Sitting and relaxing", "Lying down", "Walking", "Climbing stairs",
    "Waist bends forward", "Frontal elevation of arms", "Knees bending", "Cycling",
    "Jogging", "Running", "Jump front & back",
)

MARKER = "<|sensor_embed_placeholder|>"

SYSTEM_TEXT = (
    "Classify one MHEALTH human activity from a 2-second motion window. The valid codes are: "
    + "; ".join(f"activity_{i:02d} = {name}" for i, name in enumerate(CLASS_NAMES, 1))
    + ". Return exactly one code activity_01 through activity_12, with no explanation."
)

USER_TEXT = (
    "Sampling rate 50 Hz; 100 samples (2 seconds). Channels in order: chest accelerometer X/Y/Z; "
    "left ankle accelerometer X/Y/Z; left ankle gyroscope X/Y/Z; right lower arm accelerometer X/Y/Z; "
    "right lower arm gyroscope X/Y/Z. Values are train-normalized UCI columns and are unconverted. "
    "Ten ordered learned embeddings follow, each for ten consecutive samples over all 15 channels: "
    + MARKER + "\nUse the preceding sensor sequence to select one code from the class map."
)

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
tokenizer.add_special_tokens({"additional_special_tokens": [MARKER]})
messages = [{"role": "system", "content": SYSTEM_TEXT}, {"role": "user", "content": USER_TEXT}]
PREFIX = tokenizer.apply_chat_template(messages, tokenize=True, return_dict=False, add_generation_prompt=True, enable_thinking=True)
MARKER_POSITION = PREFIX.index(tokenizer.convert_tokens_to_ids(MARKER))
END_ID = tokenizer.convert_tokens_to_ids("<turn|>")
TARGETS = {}
prefix_len = len(PREFIX)
for label in range(1, 13):
    full = tokenizer.apply_chat_template(messages + [{"role": "assistant", "content": f"activity_{label:02d}"}], tokenize=True, return_dict=False, enable_thinking=True)
    if full[:prefix_len] != PREFIX:
        raise ValueError("学習と生成のprefixが一致しません。")
    answer = full[prefix_len:]
    TARGETS[label] = answer[:answer.index(END_ID)+1]  # 終端後の改行は学習しない。

## 2. 波形を入力できるようにする

100×15を10×150へ並べ、共有Linear150→3840で投影します。FP32のprojector出力に√(3840/150)を掛け、
Gemmaのdtypeへ変換してuser turnへ挿入します。

train lossはクラス重み付きの回答＋終端CE。valは終端を除き、各クラスの平均が同じ重みになるよう集計します。
valの `N/(12×クラス件数)` は、Trainerの全窓平均をクラス平均へ変換する係数です。

In [8]:
class SensorGemma(Gemma4UnifiedForConditionalGeneration):
    accepts_loss_kwargs = False

    def prepare_inputs_for_generation(self, input_ids, inputs_embeds=None, **kwargs):
        return super().prepare_inputs_for_generation(input_ids, inputs_embeds=inputs_embeds, **kwargs)

    def __init__(self, config):
        super().__init__(config)
        self.sensor_projector = nn.Linear(PATCH_WIDTH, config.text_config.hidden_size)

    def build_inputs_embeds(self, waveforms, answer_ids=()):
        embed_tokens = self.get_input_embeddings()
        prefix_ids = torch.tensor(
            [PREFIX[:MARKER_POSITION]], device=self.device)
        suffix_ids = torch.tensor(
            [PREFIX[MARKER_POSITION + 1:] + list(answer_ids)],
            device=self.device)

        # 1窓を10個のpatchへ分け、FP32で線形変換する。
        with torch.autocast(self.device.type, enabled=False):
            patches = waveforms.float().reshape(1, -1, PATCH_WIDTH)
            sensor_embeds = self.sensor_projector(patches)
            sensor_embeds = sensor_embeds * math.sqrt(
                embed_tokens.weight.shape[1] / PATCH_WIDTH)

        sensor_embeds = sensor_embeds.to(embed_tokens.weight.dtype)
        return torch.cat(
            (embed_tokens(prefix_ids), sensor_embeds,
             embed_tokens(suffix_ids)), dim=1)

    def forward(self, input_ids=None, attention_mask=None, inputs_embeds=None,
                waveforms=None, labels=None, logits_to_keep=0, **kwargs):
        if waveforms is None:
            return super().forward(input_ids=input_ids, attention_mask=attention_mask, inputs_embeds=inputs_embeds,
                                   labels=labels, logits_to_keep=logits_to_keep, **kwargs)
        label = int(labels[0])
        target = torch.tensor(TARGETS[label], device=self.device)
        embeddings = self.build_inputs_embeds(waveforms, target.tolist())
        positions = torch.arange(embeddings.shape[1]-len(target)-1, embeddings.shape[1]-1, device=self.device)
        kwargs.update(inputs_embeds=embeddings, attention_mask=torch.ones(embeddings.shape[:2],device=self.device),
                      logits_to_keep=positions, use_cache=False, return_dict=True)
        logits = super().forward(**kwargs).logits[0].float()
        ce = F.cross_entropy(logits,target,reduction='none')
        loss = ce.mean()*TRAIN_WEIGHTS[label-1] if self.training else ce[:-1].mean()*VAL_WEIGHTS[label-1]
        if not torch.isfinite(loss):
            raise FloatingPointError('回答lossが非有限です。')
        return {'loss': loss, 'answer_ce': ce[:-1].mean().detach(), 'end_ce': ce[-1].detach()}

In [9]:
base_model = SensorGemma.from_pretrained(MODEL_ID, revision=REVISION, dtype=torch.bfloat16, attn_implementation="sdpa", device_map={"": 0})
base_model.sensor_projector.float()
set_seed(42)
lora_config = LoraConfig(
    r=4, lora_alpha=8, lora_dropout=0.0,
    bias="none", task_type="CAUSAL_LM",
    target_modules=r".*language_model\..*\.(q_proj|v_proj)",
    modules_to_save=["sensor_projector"],
)
model = get_peft_model(base_model, lora_config)
initial_projector = nn.Linear(
    base_model.sensor_projector.in_features, base_model.sensor_projector.out_features,
    dtype=torch.float32, device="cpu")
nn.init.normal_(initial_projector.weight, std=0.02)  # projectorはseed 42でCPU初期化する。
nn.init.zeros_(initial_projector.bias)
model.base_model.model.sensor_projector.modules_to_save["default"].load_state_dict(initial_projector.state_dict())
model.print_trainable_parameters()

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

[transformers] SensorGemma LOAD REPORT from: google/gemma-4-12B-it
Key                     | Status  | 
------------------------+---------+-
sensor_projector.bias   | MISSING | 
sensor_projector.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 3,176,704 || all params: 11,963,486,720 || trainable%: 0.0266


### 生成評価の小さな補助関数

各被験者・各クラスの非重複窓から1/4位置と3/4位置を選びます。valは24例、testは72例です。
生成は公式 `generate` のgreedy、最大24tokens。終端なしはtruncated、指定形式以外はinvalidで、どちらも不正解です。
zeroは正規化後の0波形、swapは同じ被験者の次クラス（12→1）の同じ選択位置の波形です。

In [10]:
def generation_windows(subjects: tuple) -> list:
    selected = []
    for subject in subjects:
        candidates = windows(subject, 100)
        for label in range(1, 13):
            group = [w for w in candidates if w[1] == label]
            selected.extend((group[len(group)//4], group[3*len(group)//4]))
    return selected

def parse_output(ids: list[int]) -> tuple[int, str]:
    if not ids or ids[-1] != END_ID:
        return 0, "truncated"
    text = tokenizer.decode(ids[:-1], skip_special_tokens=False)
    if re.match(r"^<\|channel>thought(?:\s|$)", text):
        thought, separator, text = text.partition("<channel|>")
        if not separator or re.search(r"<[^>]*>", thought.removeprefix("<|channel>thought")):
            return 0, "invalid"
    match = re.fullmatch(r"activity_(0[1-9]|1[0-2])", text.strip())
    return (int(match[1]), "valid") if match else (0, "invalid")

@torch.inference_mode()
def predict(selected: list, condition: str = "normal") -> pl.DataFrame:
    model.eval()
    rows = []
    generation = GenerationConfig(max_new_tokens=24, do_sample=False, use_cache=True,
                                  eos_token_id=END_ID, pad_token_id=tokenizer.pad_token_id)
    for index, (subject, label, start) in enumerate(selected):
        donor = (subject, label, start)
        if condition == "swap":
            donor = [w for w in selected if w[0] == subject and w[1] == label % 12 + 1][index % 2]
        wave = torch.as_tensor(preprocess_window(donor)["waveforms"], device=model.device).unsqueeze(0)
        if condition == "zero":
            wave = torch.zeros_like(wave)
        with torch.autocast(model.device.type, dtype=torch.bfloat16, enabled=model.device.type == "cuda"):
            embeddings = model.get_base_model().build_inputs_embeds(wave)
            ids = model.generate(inputs_embeds=embeddings, attention_mask=torch.ones(embeddings.shape[:2], device=model.device),
                                 generation_config=generation)[0].tolist()
            losses = model(waveforms=wave, labels=torch.tensor([label], device=model.device))
        prediction, status = parse_output(ids)
        rows.append(dict(subject=subject, label=label, start=start, donor_label=donor[1], condition=condition,
                         prediction=prediction, status=status, completed=END_ID in ids, generated_tokens=len(ids),
                         raw=tokenizer.decode(ids, skip_special_tokens=False), answer_ce=float(losses["answer_ce"]), end_ce=float(losses["end_ce"])))
    return pl.DataFrame(rows)

## 3. Trainerで学習する

各epochの終了時にvalidation lossを評価し、LoRAとprojectorを保存します。
validation lossの改善幅が0.001を超えない状態が2回続くと早期終了し、学習終了時にbestを復元します。


In [11]:
class TrainingWindows(torch.utils.data.IterableDataset):
    epoch = 0
    def __len__(self):
        return len(train_meta)
    def set_epoch(self, epoch):
        self.epoch = epoch
    def __iter__(self):
        ordered = sorted(train_meta, key=lambda w: hashlib.sha256(f"42|{self.epoch+1}|{w[0]}|{w[2]}|{w[1]}".encode()).digest())
        return map(preprocess_window, ordered)

class SensorTrainer(Trainer):
    def evaluate(self, *args, **kwargs):
        metrics = super().evaluate(*args, **kwargs)
        predictions = predict(generation_windows((10,)))
        if not hasattr(self, "validation_predictions"):
            self.validation_predictions = {}
        self.validation_predictions[round(self.state.epoch or 0)] = predictions
        self.log({"eval_generation_accuracy": float((predictions.get_column("prediction") == predictions.get_column("label")).mean())})
        return metrics

    def _determine_best_metric(self, metrics, trial):
        best = self.state.best_metric
        if best is not None and metrics["eval_loss"] >= best - 0.001:
            return False
        return super()._determine_best_metric(metrics, trial)

In [12]:
training_args = TrainingArguments(output_dir=str(OUTPUT), num_train_epochs=10,
    per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=1e-4, weight_decay=0.01, max_grad_norm=1., warmup_steps=151, lr_scheduler_type="cosine",
    bf16=True, gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=2, save_only_model=True,
    load_best_model_at_end=True, metric_for_best_model="loss", greater_is_better=False,
    prediction_loss_only=True, label_names=["labels"], logging_steps=1, report_to="none", seed=42, full_determinism=True)


In [13]:
trainer = SensorTrainer(
    model=model, args=training_args,
    train_dataset=TrainingWindows(),
    eval_dataset=[preprocess_window(window) for window in val_meta],
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=2,
        early_stopping_threshold=0.001)],
    optimizer_cls_and_kwargs=(torch.optim.AdamW, {
        "lr": 1e-4, "betas": (0.9, 0.999), "eps": 1e-8,
        "foreach": False, "fused": False}),
)

trainer.create_optimizer()
for group in trainer.optimizer.param_groups:
    group["initial_lr"] = 1e-4
trainer.lr_scheduler = get_cosine_schedule_with_warmup(trainer.optimizer, 151, 5030, last_epoch=0)
initial_metrics = trainer.evaluate()
trainer.train()
trainer.save_model(str(OUTPUT / "best"))
tokenizer.save_pretrained(OUTPUT / "best")

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/linear.py:134: UserWarning: Requested unified CUBLASLT workspace size of 1048576 bytes exceeds CUBLAS workspace size of 131072 bytes. Please increase CUBLAS workspace size via CUBLAS_WORKSPACE_CONFIG or decrease requested CUBLASLT_WORKSPACE_SIZE. Otherwise CUBLASLT workspace size will be limited to the CUBLAS workspace size. (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:228.)
  return F.linear(input, self.weight, self.bias)


Training Loss,Validation Loss,Epoch
No log,7.363072,0


[transformers] The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Epoch,Training Loss,Validation Loss
1,0.320187,0.268324


('gemma4_mhealth_run/best/tokenizer_config.json',
 'gemma4_mhealth_run/best/chat_template.jinja',
 'gemma4_mhealth_run/best/tokenizer.json')

最大10epochs、warmup151/全5030updates、patience2、min_delta0.001です。
公式schedulerの `last_epoch=0` で、最初の更新をLR=1e-4/151とする既定の添字に揃えています。
trainとvalではlossの定義が異なるため、次の曲線は別パネルで見ます。stepはoptimizer更新回数、epochは学習データの周回です。

## 4. bestで推論・評価する

学習中は毎epochにval24例を生成し、bestは329窓のval lossで選びます。
以下ではbestのvalとtest72例をnormal/zero/swapで推論し、accuracy・macro F1・balanced accuracy・macro answer CE・invalid数・truncated数を条件ごとに標準出力します。normalからの予測変更数とaccuracy差も表示します。
評価セルを再実行すると再生成します。

In [14]:
def summarize(table: pl.DataFrame, name: str, truth: str = "label") -> dict:
    truth_values = table.get_column(truth).to_list()
    predictions = table.get_column("prediction").to_list()
    report = classification_report(truth_values, predictions, labels=range(1, 13), output_dict=True, zero_division=0)
    macro_answer_ce = table.group_by("label").agg(pl.col("answer_ce").mean()).get_column("answer_ce").mean()
    return dict(count=len(table), correct=sum(a == b for a, b in zip(truth_values, predictions)),
                accuracy=sum(a == b for a, b in zip(truth_values, predictions)) / len(table),
                macro_f1=report["macro avg"]["f1-score"], balanced_accuracy=report["macro avg"]["recall"],
                macro_answer_ce=macro_answer_ce,
                invalid=table.filter(pl.col("status") == "invalid").height,
                truncated=table.filter(pl.col("status") == "truncated").height)

updates_per_epoch = math.ceil(len(train_meta) / training_args.gradient_accumulation_steps)
best_epoch = trainer.state.best_global_step // updates_per_epoch
summaries = {}
for split, subjects in [("val", (10,)), ("test", (1, 3, 6))]:
    selected = generation_windows(subjects)
    normal = trainer.validation_predictions[best_epoch] if split == "val" else predict(selected)
    for condition in ("normal", "zero", "swap"):
        table = normal if condition == "normal" else predict(selected, condition)
        name = f"{split}_{condition}"
        summaries[name] = summarize(table, name)
        summaries[name]["changed_predictions"] = int((table.get_column("prediction") != normal.get_column("prediction")).sum())
        summaries[name]["accuracy_delta_from_normal"] = summaries[name]["accuracy"] - float((normal.get_column("prediction") == normal.get_column("label")).mean())
        if condition == "swap":
            summaries[name]["donor_metrics"] = summarize(table, f"{name}_donor", "donor_label")
print(pl.DataFrame([{"condition": name, **{key: value for key, value in summary.items() if key != "donor_metrics"}} for name, summary in summaries.items()]))

shape: (6, 11)
┌─────────────┬───────┬─────────┬──────────┬───┬─────────┬───────────┬──────────────┬──────────────┐
│ condition   ┆ count ┆ correct ┆ accuracy ┆ … ┆ invalid ┆ truncated ┆ changed_pred ┆ accuracy_del │
│ ---         ┆ ---   ┆ ---     ┆ ---      ┆   ┆ ---     ┆ ---       ┆ ictions      ┆ ta_from_norm │
│ str         ┆ i64   ┆ i64     ┆ f64      ┆   ┆ i64     ┆ i64       ┆ ---          ┆ al           │
│             ┆       ┆         ┆          ┆   ┆         ┆           ┆ i64          ┆ ---          │
│             ┆       ┆         ┆          ┆   ┆         ┆           ┆              ┆ f64          │
╞═════════════╪═══════╪═════════╪══════════╪═══╪═════════╪═══════════╪══════════════╪══════════════╡
│ val_normal  ┆ 24    ┆ 13      ┆ 0.541667 ┆ … ┆ 0       ┆ 0         ┆ 0            ┆ 0.0          │
│ val_zero    ┆ 24    ┆ 2       ┆ 0.083333 ┆ … ┆ 0       ┆ 0         ┆ 22           ┆ -0.458333    │
│ val_swap    ┆ 24    ┆ 2       ┆ 0.083333 ┆ … ┆ 0       ┆ 0         ┆ 21   

## 参考実装とライセンス

- [Google Gemma cookbook](https://github.com/google-gemma/cookbook/blob/5c52c7250feff52e0a7cc526bf32edd646bc3d6b/docs/core/huggingface_text_finetune_qlora.ipynb)（Apache-2.0）：commit `5c52c7250feff52e0a7cc526bf32edd646bc3d6b`。
- [SensorLLM](https://github.com/cruiseresearchgroup/SensorLLM/blob/ddd17fecb508fcfa5ae156dbd37902e3ddbf652a/mhealth_stage2.ipynb)（ソースコードMIT）：commit `ddd17fecb508fcfa5ae156dbd37902e3ddbf652a`。列選択・連続区間・窓切りを改変利用。
- Transformers 5.20.2、PEFT 0.21.2、Accelerate 1.15.0（Apache-2.0）。

- [MHEALTH / UCI](https://archive.ics.uci.edu/dataset/319/mhealth%2Bdataset)はCC BY 4.0。固定revisionのモデルカードはApache-2.0を示す。現行の[モデルカード](https://huggingface.co/google/gemma-4-12B-it)と[Gemma 4ライセンス](https://ai.google.dev/gemma/apache_2)も確認する。

SensorLLM由来コードのライセンス：
Google cookbook由来の著作権表示：Copyright 2025 Google LLC. Licensed under the Apache License, Version 2.0. 上記cookbookの学習例を波形入力へ変更しています。

MIT License

Copyright (c) 2024 cruiseresearchgroup

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.